In [1]:
import h5py
import numpy as np

PR = 0  # Change as needed
last_n = 78000  # number of last time steps to read

# File paths
u_file = f"/data92/PeterChang/Dycore_data/PR{PR}/u/PR{PR}_500_20000day_u_6hourly.h5"
v_file = f"/data92/PeterChang/Dycore_data/PR{PR}/v/PR{PR}_500_20000day_v_6hourly.h5"
t_file = f"/data92/PeterChang/Dycore_data/PR{PR}/t/PR{PR}_500_20000day_t_6hourly.h5"
w_file = f"/data92/PeterChang/Dycore_data/PR{PR}/w/PR{PR}_500_20000day_w_6hourly.h5"
ps_file = f"/data92/PeterChang/Dycore_data/PR{PR}/ps/PR{PR}_500_20000day_ps_6hourly.h5"

# Read only the last `last_n` time steps of the main dataset in each file
def read_last_timesteps(file_path, last_n):
    with h5py.File(file_path, 'r') as f:
        # Assume there's only one dataset or we know the key
        data_key = list(f.keys())[0]  # e.g., 'u'
        dset = f[data_key]
        total_time = dset.shape[0]
        return dset[-last_n:, ...]  # last 5000 time steps

# Load data slices
u_last = read_last_timesteps(u_file, last_n)
v_last = read_last_timesteps(v_file, last_n)
t_last = read_last_timesteps(t_file, last_n)
w_last = read_last_timesteps(w_file, last_n)
ps_last = read_last_timesteps(ps_file, last_n)  # shape likely (time, y, x)

# Confirm shapes (optional)
print("u:", u_last.shape)
print("v:", v_last.shape)
print("t:", t_last.shape)
print("w:", w_last.shape)
print("ps:", ps_last.shape)


u: (78000, 20, 64, 128)
v: (78000, 20, 64, 128)
t: (78000, 20, 64, 128)
w: (78000, 20, 64, 128)
ps: (78000, 1, 64, 128)


In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

# Constants
a = 6.371e6       # Earth radius (m)
p0 = 100000       # Reference pressure (Pa)
R = 287.0         # Gas constant for dry air (J/kg/K)
cp = 1004.0       # Specific heat at constant pressure (J/kg/K)

# Configuration
# PR = 20
# last_n = 5000
lev = np.linspace(1, 0.1, 20)  # sigma levels

# File paths
base_path = f"/data92/PeterChang/Dycore_data/PR{PR}/"
u_file = base_path + f"u/PR{PR}_500_20000day_u_6hourly.h5"
v_file = base_path + f"v/PR{PR}_500_20000day_v_6hourly.h5"
t_file = base_path + f"t/PR{PR}_500_20000day_t_6hourly.h5"
ps_file = base_path + f"ps/PR{PR}_500_20000day_ps_6hourly.h5"

def read_last_timesteps(file_path, last_n):
    with h5py.File(file_path, 'r') as f:
        key = list(f.keys())[0]
        return f[key][-last_n:]

# Load data
u_last = read_last_timesteps(u_file, last_n)
v_last = read_last_timesteps(v_file, last_n)
t_last = read_last_timesteps(t_file, last_n)
ps_last = read_last_timesteps(ps_file, last_n)[:, 0, :, :]  # (time, lat, lon)

# Compute potential temperature
def compute_theta(T, ps, sigma):
    ps_expanded = np.repeat(ps[:, None, :, :], T.shape[1], axis=1)
    p = sigma[None, :, None, None] * ps_expanded
    theta = T * (p0 / p) ** (R / cp)
    return theta

# Zonal and time mean & anomalies
def mean_and_perturb(field):
    zonal_mean = np.mean(field, axis=-1)
    time_mean = np.mean(zonal_mean, axis=0)
    anomaly = field - zonal_mean[..., None]
    return time_mean, anomaly

# Compute means and perturbations
u_bar, u_prime = mean_and_perturb(u_last)
v_bar, v_prime = mean_and_perturb(v_last)
t_bar, t_prime = mean_and_perturb(t_last)

theta = compute_theta(t_last, ps_last, lev)
theta_bar, theta_prime = mean_and_perturb(theta)

# Compute flux terms
uv_prime = np.mean(u_prime * v_prime, axis=(0, 3))         # (lev, lat)
vtheta_prime = np.mean(v_prime * theta_prime, axis=(0, 3)) # (lev, lat)
dtheta_dsigma = np.gradient(theta_bar, lev, axis=0)
dtheta_dsigma[np.abs(dtheta_dsigma)<50] = np.nan

# Latitude and geometry
lat_deg = np.linspace(-90, 90, 64)
lat_rad = np.radians(lat_deg)
cos_phi = np.cos(lat_rad)

# EP Flux
F_phi = -a * cos_phi[None, :] * uv_prime
F_sigma = a * cos_phi[None, :] * np.sin(lat_rad)[None, :]* vtheta_prime / dtheta_dsigma

# Plot
Lat, Sigma = np.meshgrid(lat_deg, lev)
plt.figure(figsize=(10, 6))
plt.quiver(Lat[1:,:], Sigma[1:,:], F_phi[1:,:], -F_sigma[1:,:]*1000)
plt.xlim([0,90])
plt.gca().invert_yaxis()
plt.xlabel('Latitude (deg)')
plt.ylabel('Sigma')
plt.title('Time-Mean Zonal-Mean EP Flux Vectors')
plt.grid()
plt.tight_layout()
plt.show()


In [ ]:
print(np.max(np.abs(dtheta_dsigma)))
Lat, Sigma = np.meshgrid(lat_deg, lev)
plt.figure(figsize=(10, 6))
cs=plt.contourf(Lat[1:,:], Sigma[1:,:], dtheta_dsigma[1:,:]) 
#plt.quiver(Lat[1:,:], Sigma[1:,:], F_phi[1:,:], -F_sigma[1:,:]*1000)
plt.quiver(Lat[1:,:], Sigma[1:,:], F_phi[1:,:], -F_sigma[1:,:]*100)
plt.contour(Lat[1:,:], Sigma[1:,:], u_bar[1:,:])

plt.colorbar(cs)
plt.xlim([0,90])
plt.gca().invert_yaxis()
plt.xlabel('Latitude (deg)')
plt.ylabel('Sigma')
plt.title('Time-Mean Zonal-Mean EP Flux Vectors')
plt.grid()
plt.tight_layout()
plt.show()